In [ ]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading on:   0%|          | 0/1500 [00:00<?, ?it/s]

Loading stop: 100%|██████████| 1500/1500 [00:06<00:00, 249.70it/s]



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:02<00:00, 159.97it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


In [43]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading stop: 100%|██████████| 1500/1500 [00:05<00:00, 252.66it/s]



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:01<00:00, 266.51it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


RandomForest

In [44]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# import m2cgen as m2c
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for a TINY Random Forest model
def objective(trial):
    param = {
        # STRICT LIMITS: Keeping trees small for the MCU's RAM
        'n_estimators': trial.suggest_int('n_estimators', 10, 30), 
        'max_depth': trial.suggest_int('max_depth', 2, 10), 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for a lightweight model...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final Random Forest model
print("\nTraining the final Random Forest model...")
final_model = RandomForestClassifier(**study.best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export the trained model parameters as lightweight arrays for MicroPython
print("\nExporting tree parameters (arrays) for memory-efficient Edge Deployment...")

trees_dump = []
for estimator in final_model.estimators_:
    tree = estimator.tree_
    
    # In Random Forest, tree.value stores the class counts at each node
    # We convert this to the actual predicted class index for leaf nodes
    node_classes = [int(np.argmax(val)) for val in tree.value]
    
    tree_data = {
        'left': tree.children_left.tolist(),
        'right': tree.children_right.tolist(),
        'feature': tree.feature.tolist(),
        # Round thresholds to 4 decimals to save storage space
        'threshold': [round(t, 4) for t in tree.threshold.tolist()], 
        'classes': node_classes
    }
    trees_dump.append(tree_data)

# Save to a Python file as a simple list of dictionaries
output_file = os.path.join(OUTPUT_PATH, 'model_data_randomForest.py')
with open(output_file, 'w') as f:
    f.write("# Auto-generated Random Forest parameters for Edge AI\n")
    f.write("TREES = [\n")
    for t in trees_dump:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"Model parameters successfully exported to: {output_file}")

[I 2026-06-28 16:24:46,449] A new study created in memory with name: no-name-419168bc-02b3-4029-842b-e7ef0964247b
[I 2026-06-28 16:24:46,614] Trial 0 finished with value: 0.7881567973311092 and parameters: {'n_estimators': 30, 'max_depth': 9}. Best is trial 0 with value: 0.7881567973311092.


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for a lightweight model...


[I 2026-06-28 16:24:46,702] Trial 1 finished with value: 0.7489574645537949 and parameters: {'n_estimators': 10, 'max_depth': 9}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:46,881] Trial 2 finished with value: 0.7848206839032527 and parameters: {'n_estimators': 27, 'max_depth': 10}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,003] Trial 3 finished with value: 0.7514595496246872 and parameters: {'n_estimators': 23, 'max_depth': 7}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,098] Trial 4 finished with value: 0.6630525437864887 and parameters: {'n_estimators': 22, 'max_depth': 2}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,202] Trial 5 finished with value: 0.6630525437864887 and parameters: {'n_estimators': 23, 'max_depth': 2}. Best is trial 0 with value: 0.7881567973311092.
[I 2026-06-28 16:24:47,297] Trial 6 finished with value: 0.7731442869057548 and parameters: {'n_estimators': 


Best parameters found: {'n_estimators': 28, 'max_depth': 9}
Best cross-validation accuracy: 79.15%

Training the final Random Forest model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.78      0.79      0.78       299
         off       0.80      0.80      0.80       300
        stop       0.90      0.84      0.87       300
     unknown       0.70      0.74      0.72       300

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199


Exporting tree parameters (arrays) for memory-efficient Edge Deployment...
Model parameters successfully exported to: ../Output\model_data_randomForest.py


LogisticRegression

In [45]:
import numpy as np
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define the Optuna objective function for Logistic Regression
def objective(trial):
    param = {
        # 'C' is the inverse of regularization strength. 
        # Smaller values specify stronger limits/constraints on the weights.
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        
        # 'solver' is the algorithm used to optimize the weights
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'saga']),
        
        # Max_iter is kept high to allow the mathematical solver to fully converge
        'max_iter': 2000, 
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = LogisticRegression(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Run the hyperparameter optimization
print("\nStarting Optuna optimization for Logistic Regression...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the final model using the exact optimized parameters
print("\nTraining the final Logistic Regression model...")
final_model = LogisticRegression(**study.best_params, max_iter=2000, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")

# Dynamically extract classes
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Export weights and intercepts to a tiny Python file
print("\nExporting model parameters (Weights & Intercepts) for Edge Deployment...")

weights = final_model.coef_.tolist()
intercepts = final_model.intercept_.tolist()

output_file = os.path.join(OUTPUT_PATH, 'model_data_regression.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Lightweight Model for Keyword Spotting\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # We round the numbers to 4 decimal places to save even more space on the MCU
    f.write("WEIGHTS = [\n")
    for class_weights in weights:
        rounded_weights = [round(w, 4) for w in class_weights]
        f.write(f"    {rounded_weights},\n")
    f.write("]\n\n")
    
    rounded_intercepts = [round(i, 4) for i in intercepts]
    f.write(f"INTERCEPTS = {rounded_intercepts}\n")

print(f"Model successfully exported to: {output_file}")
print("Done! The model_data.py is now highly optimized and incredibly lightweight.")

[I 2026-06-28 16:25:44,984] A new study created in memory with name: no-name-0c975743-a44a-4ddb-8839-808d10d11ce2


Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for Logistic Regression...


c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-28 16:25:46,756] Trial 0 finished with value: 0.7948290241868223 and parameters: {'C': 16.896513260974356, 'solver': 'saga'}. Best is trial 0 with value: 0.7948290241868223.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
[I 2026-06-28 16:25:48,773] Trial 1 finished with value: 0.7948290241868223 and parameters: {'C': 0.422188412563391, 'solver': 'saga'}. Best is trial 0 with value: 0.7948290241868223.
c:\Users\User\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no 


Best parameters found: {'C': 16.896513260974356, 'solver': 'saga'}
Best cross-validation accuracy: 79.48%

Training the final Logistic Regression model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.79      0.82      0.80       299
         off       0.82      0.79      0.80       300
        stop       0.88      0.82      0.85       300
     unknown       0.70      0.75      0.73       300

    accuracy                           0.79      1199
   macro avg       0.80      0.79      0.80      1199
weighted avg       0.80      0.79      0.80      1199


Exporting model parameters (Weights & Intercepts) for Edge Deployment...
Model successfully exported to: ../Output\model_data_regression.py
Done! The model_data.py is now highly optimized and incredibly lightweight.


In [47]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_regression as model_data

def predict_audio_class(mfcc_features):
    """
    Computes the dot product of features and weights for each class,
    adds the intercept, and returns the class with the highest score.
    """
    num_classes = len(model_data.INTERCEPTS)
    num_features = len(mfcc_features)
    
    scores = []
    
    # Calculate score for each class
    for i in range(num_classes):
        class_score = model_data.INTERCEPTS[i]
        
        # Dot product: multiply each feature by its corresponding weight
        for j in range(num_features):
            class_score += mfcc_features[j] * model_data.WEIGHTS[i][j]
            
        scores.append(class_score)
        
    # Find the index of the highest score (Argmax)
    best_class_idx = 0
    max_score = scores[0]
    for i in range(1, num_classes):
        if scores[i] > max_score:
            max_score = scores[i]
            best_class_idx = i
            
    return best_class_idx

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = X_test[i] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])

IndexError: list index out of range

XGBoost

In [57]:
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import json

DATA_PATH = '../Data/'
OUTPUT_PATH = '../Output'

# 1. Load the pre-processed features and labels
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Define Optuna objective for XGBoost
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'max_depth': trial.suggest_int('max_depth', 2, 3), 
        'n_estimators': trial.suggest_int('n_estimators', 2, 5), 
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    return accuracy_score(y_test, preds)

# 3. Optimize Hyperparameters
print("\nStarting Optuna optimization for XGBoost...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\nBest parameters found:", study.best_params)
print(f"Best cross-validation accuracy: {study.best_value * 100:.2f}%")

# 4. Train the Final XGBoost Model
print("\nTraining the final XGBoost model...")
best_params = study.best_params
best_params['objective'] = 'multi:softprob'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

# Evaluate the final model
final_preds = final_model.predict(X_test)
print("\nClassification Report on Test Data:")
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. CUSTOM EXPORT: Bypass m2cgen and extract trees directly
print("\nExporting XGBoost trees manually for ultra-lightweight Edge Deployment...")

booster = final_model.get_booster()
# Dump all trees to JSON format
trees_json = booster.get_dump(dump_format='json')

parsed_trees = []
for tree_str in trees_json:
    tree_dict = json.loads(tree_str)
    
    # Recursive function to find the maximum node ID to size our arrays
    def get_max_nodeid(node):
        max_id = node.get('nodeid', 0)
        if 'children' in node:
            for child in node['children']:
                max_id = max(max_id, get_max_nodeid(child))
        return max_id
        
    size = get_max_nodeid(tree_dict) + 1
    
    # Initialize flattened arrays for MicroPython
    features = [-1] * size
    thresholds = [0.0] * size
    lefts = [-1] * size
    rights = [-1] * size
    values = [0.0] * size
    
    # Recursive function to parse JSON into flat arrays
    def parse_node(node):
        nid = node['nodeid']
        if 'leaf' in node:
            values[nid] = round(node['leaf'], 4)
        else:
            # XGBoost features are formatted as 'f0', 'f1', etc. We extract the integer.
            feat_str = node['split']
            features[nid] = int(feat_str.replace('f', ''))
            thresholds[nid] = round(node['split_condition'], 4)
            lefts[nid] = node['yes']
            rights[nid] = node['no']
            for child in node['children']:
                parse_node(child)
                
    parse_node(tree_dict)
    
    parsed_trees.append({
        'features': features,
        'thresholds': thresholds,
        'left': lefts,
        'right': rights,
        'values': values
    })

# Save the parsed structure to a Python file
output_file = os.path.join(OUTPUT_PATH, 'model_data_xgb.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated Custom XGBoost Export for MicroPython\n\n")
    f.write(f"CLASSES = {target_names}\n")
    f.write(f"NUM_CLASS = {len(target_names)}\n\n")
    f.write("TREES = [\n")
    for t in parsed_trees:
        f.write(f"    {t},\n")
    f.write("]\n")

print(f"XGBoost Model successfully exported to: {output_file}")

[I 2026-06-28 16:45:55,963] A new study created in memory with name: no-name-8ea389fb-49c9-444c-9580-6d5c3cc6bec4
[I 2026-06-28 16:45:56,028] Trial 0 finished with value: 0.6705587989991659 and parameters: {'max_depth': 3, 'n_estimators': 5, 'learning_rate': 0.1629526582033109}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,071] Trial 1 finished with value: 0.6705587989991659 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rate': 0.19436206613917192}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,110] Trial 2 finished with value: 0.609674728940784 and parameters: {'max_depth': 2, 'n_estimators': 5, 'learning_rate': 0.06295034245628427}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,138] Trial 3 finished with value: 0.6413678065054211 and parameters: {'max_depth': 3, 'n_estimators': 2, 'learning_rate': 0.04289356250519018}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,159] 

Loaded Data -> X shape: (5995, 39), y shape: (5995,)

Starting Optuna optimization for XGBoost...


[I 2026-06-28 16:45:56,181] Trial 5 finished with value: 0.6021684737281068 and parameters: {'max_depth': 2, 'n_estimators': 2, 'learning_rate': 0.23680486591372515}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,215] Trial 6 finished with value: 0.6130108423686406 and parameters: {'max_depth': 2, 'n_estimators': 4, 'learning_rate': 0.07789237481743871}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,243] Trial 7 finished with value: 0.6413678065054211 and parameters: {'max_depth': 3, 'n_estimators': 2, 'learning_rate': 0.05317258025910898}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,263] Trial 8 finished with value: 0.6055045871559633 and parameters: {'max_depth': 2, 'n_estimators': 2, 'learning_rate': 0.2559616521940972}. Best is trial 0 with value: 0.6705587989991659.
[I 2026-06-28 16:45:56,306] Trial 9 finished with value: 0.6447039199332777 and parameters: {'max_depth': 3, 'n_estimators': 4, 'learning_rat


Best parameters found: {'max_depth': 3, 'n_estimators': 3, 'learning_rate': 0.296072654468693}
Best cross-validation accuracy: 68.64%

Training the final XGBoost model...

Classification Report on Test Data:
              precision    recall  f1-score   support

          on       0.66      0.68      0.67       299
         off       0.74      0.75      0.74       300
        stop       0.83      0.69      0.75       300
     unknown       0.56      0.63      0.59       300

    accuracy                           0.69      1199
   macro avg       0.70      0.69      0.69      1199
weighted avg       0.70      0.69      0.69      1199


Exporting XGBoost trees manually for ultra-lightweight Edge Deployment...
XGBoost Model successfully exported to: ../Output\model_data_xgb.py


In [56]:
import sys
import os

sys.path.append(os.path.abspath('../Output'))

import model_data_xgb as model_data
def predict_audio_class(mfcc_features):
    """
    Traverses the exported XGBoost trees using the extracted MFCC features.
    Returns the index of the predicted class based on the highest raw score.
    """
    num_class = model_data.NUM_CLASS
    
    # Array to accumulate scores (raw logits) for each class
    class_scores = [0.0] * num_class
    
    # In multi-class XGBoost, trees are assigned to classes in a round-robin fashion.
    # Tree 0 -> Class 0, Tree 1 -> Class 1 ... Tree 4 -> Class 0, etc.
    for i, tree in enumerate(model_data.TREES):
        node = 0 # Start at the root of the tree
        
        # Traverse until a leaf is reached (marked by feature index -1)
        while tree['features'][node] != -1:
            feat_idx = tree['features'][node]
            
            # XGBoost splitting rule: strictly less than (<)
            if mfcc_features[feat_idx] < tree['thresholds'][node]:
                node = tree['left'][node]
            else:
                node = tree['right'][node]
                
        # Leaf reached: get the value and add it to the corresponding class score
        leaf_value = tree['values'][node]
        class_idx = i % num_class
        class_scores[class_idx] += leaf_value
        
    # Find the index of the highest score (Argmax)
    best_class = 0
    max_score = class_scores[0]
    for i in range(1, num_class):
        if class_scores[i] > max_score:
            max_score = class_scores[i]
            best_class = i
            
    return best_class

# --- Example Usage on Board ---
# ['on', 'off', 'stop', 'uknown']
i = 16
features = [-2.5686844e+02,  1.3959177e+02,  8.0665617e+00, -4.1448826e+01, -2.9944046e+01,  3.2988396e-01,  6.2178426e+00,  9.0643892e+00, -6.5838027e+00,  5.9275532e+00, -2.8192373e+01,  1.9626240e+01, -1.0130860e+00, -2.5470102e+02,  1.6171916e+02, -9.4717093e+00, -1.6660915e+01,  9.5316715e+00,  1.5823692e+01, -4.0831656e+00, -1.6109245e+01, -1.6403475e+01, -2.7065601e+00, -7.6615257e+00, 1.2409652e+01, -2.4190521e+01, -4.9443311e+02,  8.9877983e+01, 1.8117041e+01,  3.1036968e+01,  5.0818684e+01,  3.1483164e+01, 4.5603323e+00, -1.2117879e+01, -1.4944232e+01,  2.7241716e+00, 7.9998846e+00, -1.2753278e+01, -1.6924435e+01] # 13 features from mic
result_idx = predict_audio_class(features)
print("Predicted Word:", model_data.CLASSES[result_idx], "and actuall is:", y_test[i])


Predicted Word: off and actuall is: 1
